# Few-shot Discord event announcement (`ai_query`)

**Purpose:** Build a Japanese Discord-style event announcement using the few-shot prompt layout from `docs/few_shot_discord_event_announcement_samples.md`, then call Databricks **`ai_query`** on a Foundation Model endpoint.

**Author:** Cheng Wang  
**Contact:** cheng.wang@myteam.com  
**Date:** 2026-03-23  

## Prerequisites

- Workspace in a region where [AI Functions / `ai_query`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query) are available.
- Your user or job identity has **CAN QUERY** on the configured foundation endpoint.
- Cluster / notebook uses a **Databricks Runtime** that supports `ai_query` (and `modelParameters` if used; DBR 15.3+).
- Adjust **`FOUNDATION_ENDPOINT`**, **`TEMPERATURE`**, and **`MAX_TOKENS`** in the constants cell if your workspace uses different defaults.

## Prompt source

Static few-shot blocks are embedded below (self-contained on the cluster).  
`Source: docs/few_shot_discord_event_announcement_samples.md`


## Web app contract (later)

| Notebook widget   | Client (form / JSON)     |
|-------------------|--------------------------|
| `tone`            | Same six hyperparameters |
| `length`          | as dropdown fields       |
| `formality`       |                          |
| `emoji_density`   |                          |
| `structure`       |                          |
| `cta_strength`    |                          |
| `user_request`    | Natural-language field   |

**Server-side only (not from client):** context facts (`CONTEXT_FOR_REQUEST`), model endpoint name, `TEMPERATURE`, `MAX_TOKENS` — same as the constants in this notebook. Build the full prompt server-side, then run one `ai_query` row (or equivalent SQL / Jobs API).


In [ ]:
# --- Fixed in code (edit per environment / event) ---
# Source: docs/few_shot_discord_event_announcement_samples.md

CONTEXT_FOR_REQUEST = """- 日時: 5月24日（土）21:00
- ゲーム: Geoguessr
- 場所: TitanZz Discord
- 参加方法: この投稿にリアクション
"""

FOUNDATION_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
TEMPERATURE = 0.3
MAX_TOKENS = 2048


In [ ]:
# Widgets: six behavior hyperparameters + natural language request
dbutils.widgets.dropdown("tone", "カジュアル", ["真面目", "おふざけ", "カジュアル"], "Tone")
dbutils.widgets.dropdown("length", "medium", ["short", "medium", "long"], "Length")
dbutils.widgets.dropdown("formality", "ですます", ["ですます", "タメ口", "敬語"], "Formality")
dbutils.widgets.dropdown("emoji_density", "普通", ["なし", "少なめ", "普通", "多め"], "Emoji density")
dbutils.widgets.dropdown("structure", "箇条書き中心", ["箇条書き中心", "段落中心", "見出し＋本文"], "Structure")
dbutils.widgets.dropdown("cta_strength", "普通", ["控えめ", "普通", "強め"], "CTA strength")
dbutils.widgets.text(
    "user_request",
    "来週土曜21時の練習会告知を、カジュアルで中くらいの長さ、箇条書き中心で作って",
    "User request",
)

tone = dbutils.widgets.get("tone")
length = dbutils.widgets.get("length")
formality = dbutils.widgets.get("formality")
emoji_density = dbutils.widgets.get("emoji_density")
structure = dbutils.widgets.get("structure")
cta_strength = dbutils.widgets.get("cta_strength")
user_request = dbutils.widgets.get("user_request")


In [ ]:
# Static few-shot core (System + single-param + combined + output rules)
# Source: docs/few_shot_discord_event_announcement_samples.md

STATIC_PROMPT_CORE = '[System / Role]\nYou are a post writer. Follow predefined behavior settings and examples.\n\n[Single-parameter Examples (few-shot; 全項目)]\n- Tone examples:\n  - 真面目: 「本日20時より練習会を実施します。参加可否をご連絡ください。」\n  - おふざけ: 「今夜20時、修行の時間だ！遅刻したら伝説になります。」\n  - カジュアル: 「今日20時から練習やるよ〜！来れそうならリアクションしてね！」\n- Length examples:\n  - short: 「本日20時から練習会です。参加者はリアクションお願いします。」\n  - medium: 「本日20時から練習会を行います。初心者歓迎、途中参加OKです。参加できる方はリアクションをお願いします。」\n  - long: 「本日20時から練習会を開催します。初心者の方も安心して参加できます。途中参加・途中退出も可能です。参加予定の方は事前にリアクションをお願いします。」\n- Formality examples:\n  - ですます: 「本日20時から練習会を開催します。ご参加お待ちしています。」\n  - タメ口: 「今日20時から練習会やるよ。来れたら来て！」\n  - 敬語: 「本日20時より練習会を開催いたします。ご参加のほどよろしくお願いいたします。」\n- Emoji density examples:\n  - なし: 「本日20時から練習会を開催します。参加希望者は連絡してください。」\n  - 少なめ: 「本日20時から練習会を開催します。参加希望者は連絡してください🙂」\n  - 普通: 「本日20時から練習会です🎮 途中参加OKです✨ 参加者は連絡ください🙌」\n  - 多め: 「本日20時から練習会です🎮🔥 初心者歓迎✨ 途中参加OK🙌 参加者は連絡ください🚀」\n- Structure examples（同一内容を形式だけ変更）:\n  - 箇条書き中心:\n    - 日時: 金曜20:00\n    - 内容: 練習会\n    - 参加方法: リアクション\n  - 段落中心:\n    「金曜20:00に練習会を行います。参加希望者はこの投稿にリアクションしてください。」\n  - 見出し＋本文:\n    「### 金曜練習会のお知らせ\n    金曜20:00に練習会を行います。参加希望者はリアクションしてください。」\n- Call-to-action strength examples（末尾1文のみ変更）:\n  - 控えめ: 「参加できる方は、可能であればリアクションをお願いします。」\n  - 普通: 「参加できる方はリアクションをお願いします。」\n  - 強め: 「参加する方は今すぐリアクションしてください！」\n\n[Combined Examples（複合指定→出力結果）]\n- Case 1:\n  - Hyperparameters: Tone=カジュアル, Length=medium, Formality=ですます, Emoji density=普通, Structure=箇条書き中心, CTA=普通\n  - Context: Geoguessr, 5/24 21:00, TitanZz Discord, 参加方法はリアクション\n  - User request: 「来週土曜21時の練習会告知を、カジュアルで中くらいの長さ、箇条書き中心で作って」\n  - Output（原文）:\n    ```text\n    📢 TitanZz Geoguessr 開催決定！  🌍✨,\n    ～配信で話題の“地図当てゲーム”、みんなでワイワイやろう！～,\n     🗓️日程 ：5月24日（土）21:00～\n     🎮ゲーム ：Geoguessr（無料でプレイ！）\n     🚩場所 ：TitanZz Discordサーバー\n    \n    🔹「配信で見たけどやったことない！」って人にも体験してほしい🌱\n    🔹いまハマってる人たちも、もっと盛り上がれたら最高！🚀\n    🔹途中参加・途中離脱・観戦だけでもOK！\n    \n    👉【参加方法】\n    この投稿にリアクションするだけ🙌\n    \n    「ちょっとやってみたいかも…」って人も、「俺の地理力、見せつけるぜ！」って人も、\n    みんなでワイワイやりましょー！初参加＆エンジョイ勢も大歓迎！\n    お待ちしてます🌏🗺️✨\n    ```\n- Case 2:\n  - Hyperparameters: Tone=おふざけ, Length=long, Formality=タメ口, Emoji density=多め, Structure=見出し＋本文, CTA=強め\n  - Context: Among Us, 5/5 21:00, Discord開催\n  - User request: 「宇宙人狼大会の告知を、ノリ強めでお祭り感ある文章にして」\n  - Output（原文）:\n    ```text\n    🚨TitanZz緊急ミッション発令🚨\n    🌌Among Us宇宙人狼大会 @GW🌌\n    ～裏切り者は、誰だッ！？～\n    \n    🧢JuJu艦長からの緊急通達🧢\n    「……船内で異常反応。念のため、緊急会議を開く。」\n    \n    乗組員たちがくつろいでいたそのとき、\n    JuJu艦長が静かに、“あのボタン”に手をかけた――\n    \n    🔴緊急会議 発動🔴\n    「裏切り者は、この中にいる。」\n    \n    🛸TitanZz宇宙船、5月5日 21:00 出航予定\n    信じ合う（フリをする）仲間たちと、疑心暗鬼の旅へ。\n    \n    🎮 ゲーム：Among Us（スマホ・PCどちらでもOK）\n    🗓️ 日程：5月5日（月）21:00～\n    📍 場所：TitanZz Discordサーバー\n    \n    🔪誰がインポスターなのか、誰がガチで頭いいのか、そして誰がただのバカなのか――\n    全てが暴かれる夜が始まる。\n    \n    🛸参加方法：この投稿にリアクションでOK！\n    \n    🌟 初心者・初見 大歓迎！\n    🌟 エンジョイ勢の参加も余裕でアリ！\n    🌟 もちろん途中参加・途中離脱OK！\n    ```\n- Case 3:\n  - Hyperparameters: Tone=真面目, Length=long, Formality=敬語, Emoji density=少なめ, Structure=見出し＋本文, CTA=控えめ\n  - Context: オフラインイベント最終連絡、会場住所あり、集合11:00\n  - User request: 「参加者向け最終連絡を、丁寧で抜け漏れなく書いて」\n  - Output（原文）:\n    ```text\n    📢 @eスポカフェ25春 参加者の皆さまへ\n    明日のオフラインイベントに関する最終連絡です！\n    \n    🎮 タイムスケジュール\n    当日のコンテンツは盛りだくさん！\n    \n    🕚 11:00 ～ 12:30\n    ・受付・フリープレイ\n    \n    🕧 12:30 ～ 14:30\n    ・VALORANT トーナメント（チーム分けは当日アナウンス予定）\n    \n    🕒 15:00 ～ 16:00\n    ・LoL「ムンドドッジボール」\n    ・Street Fighter 6 対戦会（同時進行）\n    ・Overwatch カスタムマッチ（同時進行）\n    \n    ✨すべて観戦もOK！出る人も見る人も楽しめる構成です！\n    \n    📍 場所\n    e-sportscafe LIG\n    〒210-0006 神奈川県川崎市川崎区砂子２−６−１１\n    🗺️ 公式サイト\n    🎥 行き方動画\n    \n    👥 集合時間・連絡先\n    🕚 11:00〜12:00 現地集合！\n    到着したら @Xenon か @JuJu  か @Aru か @Axela に直接 or Discordで連絡ください！\n    \n    ⚠️ 注意事項\n    ・昼食は各自で自由に（持ち込みOK！）\n    ・デバイス持ち込み可（お店の備え付けも使えます）\n    ・途中参加／途中退室OK（事前に一言いただけると助かります！）\n    \n    明日は思いっきり楽しみましょう！！\n    ご不明点があればこの投稿にリプライください\U0001fae1🔥 \n    ```\n- Case 4:\n  - Hyperparameters: Tone=カジュアル, Length=short, Formality=ですます, Emoji density=少なめ, Structure=段落中心, CTA=強め\n  - Context: 8/30-31 OECU CUP 出場告知\n  - User request: 「出場告知を短く、応援お願いを強めで」\n  - Output:\n    「8/30-31にTitanZzがOECU CUPへ出場します！🔥\n    応援してくださる方は、ぜひ拡散とリアクションをお願いします！」 \n- Case 5:\n  - Hyperparameters: Tone=カジュアル, Length=medium, Formality=タメ口, Emoji density=普通, Structure=段落中心, CTA=普通\n  - Context: OECU CUP 優勝報告\n  - User request: 「結果報告を明るく、感謝を入れて書いて」\n  - Output:\n    「🏆 OECU CUP 優勝！\n    2日間の試合を勝ち抜いて優勝できました。応援してくれたみんな本当にありがとう！🙌\n    次も頑張るので引き続きよろしく！」\n\n[Output instruction]\n- 最終結果は本文のみ出力する\n- 余計な解説・分析は出力しない\n'


In [ ]:
def build_hyperparameter_block(
    tone: str,
    length: str,
    formality: str,
    emoji_density: str,
    structure: str,
    cta_strength: str,
) -> str:
    length_help = (
        "short: 〜100字（1〜2文） / medium: 101〜300字（2〜5文） / long: 301字〜（補足・背景あり）"
    )
    emoji_help = "なし=0個 / 少なめ=1〜2個 / 普通=3〜5個 / 多め=6個以上"
    return f"""[Hyperparameter Definitions — この生成の指定値]
- Tone: {tone}
- Length: {length} ({length_help})
- Formality: {formality}
- Emoji density: {emoji_density} ({emoji_help})
- Structure: {structure}
- Call-to-action strength: {cta_strength}
"""


hyper_block = build_hyperparameter_block(
    tone, length, formality, emoji_density, structure, cta_strength
)

_split = STATIC_PROMPT_CORE.split("[Single-parameter Examples", 1)
_head = _split[0].rstrip()
_single_combined = "[Single-parameter Examples" + _split[1].rsplit("[Output instruction]", 1)[0].rstrip()
_output_instruction = (
    "[Output instruction]" + STATIC_PROMPT_CORE.rsplit("[Output instruction]", 1)[1].strip()
)

full_prompt = (
    _head
    + "\n\n"
    + hyper_block
    + "\n\n"
    + _single_combined
    + "\n\n[Context for this request]\n"
    + CONTEXT_FOR_REQUEST.strip()
    + "\n\n[User request (natural language)]\n"
    + user_request.strip()
    + "\n\n"
    + _output_instruction
)


In [ ]:
df = spark.createDataFrame([(full_prompt,)], ["full_prompt"])

# modelParameters and failOnError require DBR 15.3+; simplify if your runtime errors.
result = df.selectExpr(
    f"ai_query('{FOUNDATION_ENDPOINT}', full_prompt, "
    f"modelParameters = named_struct('temperature', CAST({TEMPERATURE} AS DOUBLE), "
    f"'max_tokens', CAST({MAX_TOKENS} AS INT)), "
    "failOnError = false) AS announcement"
)

display(result)
